### Import Libraries

In [100]:
import pandas as pd
import geopandas as gpd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

import os; os.chdir('/Users/aadityaparanjape/Downloads/IDX Exchange')

### Load Data

In [101]:
# sets for linear regression
linear_train = pd.read_csv('data/model_sets/train_set_unscaled.csv')
linear_test = pd.read_csv('data/model_sets/test_set_unscaled.csv')

In [102]:
# sets for tree models
tree_train = pd.read_csv('data/model_sets/train_set_unscaled.csv')
tree_test = pd.read_csv('data/model_sets/test_set_unscaled.csv')

### Feature Engineering

In [103]:
currentYear=2026
amenities = ['ViewYN', 'PoolPrivateYN', 'FireplaceYN', 'AttachedGarageYN']

# read GeoJSON
districts = gpd.read_file('data/enriched_sets/DistrictAreas2526_-284845464123469011.geojson')

# filer for unified districts
districts = districts[districts['DistrictType'] == 'Unified']

# keep only cols that we need and standardize 
districts = districts[['geometry', 'DistrictName']]
districts = districts.to_crs("EPSG:4326")

def add_features(df):
    df = df.copy()

    # PropertyAge
    df['PropertyAge'] = currentYear - df['YearBuilt']

    # BedBathRatio
    df['BedBathRatio'] = df['BathroomsTotalInteger'] / df['BedroomsTotal']

    # AmenityScore
    for col in amenities:
        df[col] = df[col].astype(int)
    df['AmenityScore'] = df[amenities].sum(axis=1)

    # DistrictName
    # Convert lat/long into geographic points
    points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs="EPSG:4326"
    )

    # Spatial join against Unified district polygons
    joined = gpd.sjoin(points, districts, how='left', predicate='within')

    # Clean up, keep DistrictName
    joined = joined.drop(columns=['geometry', 'index_right'])

    return joined

# add new features
linear_train = add_features(linear_train)
linear_test  = add_features(linear_test)
tree_train   = add_features(tree_train)
tree_test    = add_features(tree_test)

# save enriched datasets
linear_train.to_csv("data/enriched_sets/linear_train_enriched.csv", index=False)
linear_test.to_csv("data/enriched_sets/linear_test_enriched.csv", index=False)
tree_train.to_csv("data/enriched_sets/tree_train_enriched.csv", index=False)
tree_test.to_csv("data/enriched_sets/tree_test_enriched.csv", index=False)

### Models and Predictions

In [104]:
models = {
    'Linear': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=54),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=54)
}

# linear model
X_train_linear = linear_train.drop(columns=['logClosePrice', 'ClosePrice'])
y_train_linear = linear_train['logClosePrice']
X_test_linear = linear_test.drop(columns=['logClosePrice', 'ClosePrice'])
y_test_linear = linear_test['logClosePrice']

# tree models
X_train_tree = tree_train.drop(columns=['logClosePrice', 'ClosePrice'])
y_train_tree = tree_train['logClosePrice']
X_test_tree = tree_test.drop(columns=['logClosePrice', 'ClosePrice'])
y_test_tree = tree_test['logClosePrice']

results = {}

for name, model in models.items():
    if name == 'Linear':
        X_train, y_train = X_train_linear, y_train_linear
        X_test, y_test = X_test_linear, y_test_linear
    else:
        X_train, y_train = X_train_tree, y_train_tree
        X_test, y_test = X_test_tree, y_test_tree

    # fit
    model.fit(X_train, y_train)

    # predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # r2 score
    r2 = round(r2_score(y_test, y_pred_test), 4)
    results[name] = {'R2': r2}

results

ValueError: could not convert string to float: 'San Ramon Valley Unified'